In [1]:
%pip install -U -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
%%writefile runtime_app.py
import os
import re
import json
import logging, traceback
from typing import Dict, Any, Optional

import requests

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

# MCP (Gateway) クライアント
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

# --- ログ設定（CloudWatch に残る） ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# --- 必須: Gateway の /mcp URL ---
# 例: https://<gateway-id>.gateway.bedrock-agentcore.<region>.amazonaws.com/mcp
GATEWAY_URL = os.environ.get("GATEWAY_URL") or "https://oyaizu-gateway-dypv7pbiue.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"

# --- Bedrock モデル（リージョン明示推奨） ---
model = BedrockModel(model_id="amazon.nova-pro-v1:0", region_name="us-east-1")

# ─────────────────────────────────────────────
# ✅ 返品ポリシー用のローカルツール（安定化のため引数あり／返り値は文字列統一）
@tool(name="return_policy_simple", description="返品・保証ポリシー（簡易版）")
def get_return_policy(product_category: str = "general") -> str:
    """
    カテゴリ別の返品・保証ポリシー（簡易版）を返します。
    Args:
        product_category: 例) 'smartphones', 'laptops', 'accessories', 'general'
    """
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available",
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty",
        },
        "general": {
            "window": "30 days",
            "condition": "Original condition with all components",
            "process": "Contact technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Policies vary by region",
            "warranty": "Standard manufacturer warranty applies",
        },
    }
    cat = (product_category or "general").lower()
    policy = return_policies.get(cat, return_policies["general"])
    title = (product_category or "general").title()
    return (
        f"Return Policy - {title}:\n\n"
        f"• Return window: {policy['window']} from delivery\n"
        f"• Condition: {policy['condition']}\n"
        f"• Process: {policy['process']}\n"
        f"• Refund timeline: {policy['refund_time']}\n"
        f"• Shipping: {policy['shipping']}\n"
        f"• Warranty: {policy['warranty']}"
    )

# ─────────────────────────────────────────────
SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact
"""

# 意図判定（超軽量）
def pick_intent(user_input: str) -> str:
    text = (user_input or "").lower()
    if any(k in text for k in ["返品", "返金", "保証", "return", "warranty"]):
        return "returns"
    if "メール" in text or "email" in text or re.search(r"\bid\s*:", user_input) or re.search(r"\busername\s*:", user_input):
        return "email"
    return "general"

# email 直書き（例: "id:USER_001,username:takumi"）の抽出
def parse_id_username(text: str) -> Optional[Dict[str, str]]:
    if not text:
        return None
    m = re.search(r"id\s*:\s*([^\s,]+)", text, flags=re.I)
    n = re.search(r"username\s*:\s*([^\s,]+)", text, flags=re.I)
    if m and n:
        return {"id": m.group(1), "username": n.group(1)}
    return None

# Gateway /mcp: tools/list 事前ヘルスチェック
def gateway_health_check(url: str, auth: Optional[str]) -> None:
    headers = {"Content-Type": "application/json"}
    if auth:
        headers["Authorization"] = auth
    payload = {"jsonrpc": "2.0", "id": "health", "method": "tools/list"}
    r = requests.post(url, headers=headers, json=payload, timeout=10)
    logger.info(f"[Gateway health] status={r.status_code} body={r.text[:300]}")
    r.raise_for_status()

# Gateway /mcp: tools/call を直接叩く（モデルの ToolUse に依存しない）
def call_gateway_tool(url: str, auth: Optional[str], tool_name: str, args: Dict[str, Any]) -> str:
    headers = {"Content-Type": "application/json"}
    if auth:
        headers["Authorization"] = auth
    payload = {
        "jsonrpc": "2.0",
        "id": "call",
        "method": "tools/call",
        "params": {"name": tool_name, "arguments": args},
    }
    r = requests.post(url, headers=headers, json=payload, timeout=15)
    r.raise_for_status()
    data = r.json()
    # 返却形式はターゲット実装に依存するため防御的に
    result = data.get("result") or data
    if isinstance(result, dict):
        # content や email 等、代表的なキーを拾う
        if "email" in result:
            return str(result["email"])
        if "content" in result:
            try:
                # 文字列のときのみ
                return result["content"][0]["text"] if isinstance(result["content"], list) else str(result["content"])
            except Exception:
                return json.dumps(result, ensure_ascii=False)
        return json.dumps(result, ensure_ascii=False)
    return str(result)

# --- AgentCore Runtime App ---
app = BedrockAgentCoreApp()

@app.entrypoint
async def invoke(payload: Dict[str, Any], context=None):
    try:
        user_input = (payload or {}).get("prompt", "").strip()
        if not user_input:
            return "[bad request] 'prompt' is required"

        # 0) 環境値と呼び出しヘッダの可視化
        logger.info(f"GATEWAY_URL={GATEWAY_URL!r}")
        request_headers = getattr(context, "request_headers", {}) or {}
        auth_header = request_headers.get("Authorization")  # "Bearer eyJ..."
        logger.info(f"Authorization header present? {bool(auth_header)}")

        # 1) GATEWAY_URL チェック
        if not GATEWAY_URL or not GATEWAY_URL.endswith("/mcp"):
            return "[runtime error] GATEWAY_URL is missing or doesn't end with '/mcp'."

        # 2) ヘルスチェック
        try:
            gateway_health_check(GATEWAY_URL, auth_header)
        except Exception as he:
            logger.error("Gateway health check failed", exc_info=True)
            return f"[runtime error] MCPClientInitializationError: gateway health check failed: {he}"

        # 3) 意図でツール最小化
        intent = pick_intent(user_input)

        # 3-1) email: id/username が既に与えられた場合は「直呼び」して即返す（最も確実）
        if intent == "email":
            parsed = parse_id_username(user_input)
            if parsed:
                # Gateway 側ツール名の解決（リストから DefaultTool を探す／なければ既知名）
                tool_name = None
                # 先に MCP クライアントを開いてツール名を柔軟に特定
                mcp_client = MCPClient(lambda: streamablehttp_client(
                    url=GATEWAY_URL,
                    headers={"Authorization": auth_header} if auth_header else {}
                ))
                with mcp_client:
                    mcp_tools = mcp_client.list_tools_sync()
                    # 典型: "...___DefaultTool" を優先
                    for t in mcp_tools:
                        n = t.get("name") if isinstance(t, dict) else getattr(t, "name", None)
                        if n and n.endswith("DefaultTool"):
                            tool_name = n
                            break
                    if not tool_name and mcp_tools:
                        # 先頭をフォールバックに
                        t0 = mcp_tools[0]
                        tool_name = t0.get("name") if isinstance(t0, dict) else getattr(t0, "name", None)

                if not tool_name:
                    return "[runtime error] could not resolve MCP tool name."

                try:
                    result_text = call_gateway_tool(GATEWAY_URL, auth_header, tool_name, parsed)
                    # 期待通りの email が返らない場合もあるので、そのままテキスト返却
                    return result_text
                except Exception as e:
                    logger.error("Direct tools/call failed", exc_info=True)
                    # 失敗時は通常フローにフォールバック（下に続行）

        # 3-2) モデルに任せる場合（返品 or email（id/username 未提示））
        mcp_client = MCPClient(lambda: streamablehttp_client(
            url=GATEWAY_URL,
            headers={"Authorization": auth_header} if auth_header else {}
        ))

        with mcp_client:
            mcp_tools = mcp_client.list_tools_sync()

            if intent == "returns":
                # 返品はローカルツールのみを渡して誤選択を防ぐ
                tools = [get_return_policy]
                sys_prompt = SYSTEM_PROMPT + """
Tool selection policy:
- For any question about returns/warranty/guarantee, ALWAYS call `return_policy_simple`
  with {"product_category":"general"} unless the user specifies a category.
- Do NOT call MCP tools for policy questions.
"""
            elif intent == "email":
                # email は MCP のみ（DefaultTool）を渡して誤選択を防ぐ
                tools = mcp_tools
                sys_prompt = SYSTEM_PROMPT + """
Tool selection policy:
- To look up a user's email, ALWAYS call the MCP tool with a JSON argument:
  {"id":"<USER_ID>", "username":"<USERNAME>"}.
- If either field is missing, first ask the user for both values (ID and username), then call the tool.
- Return only the email in the final answer.
"""
            else:
                # 一般質問：最低限に留める（必要に応じて拡張）
                tools = [get_return_policy] + mcp_tools
                sys_prompt = SYSTEM_PROMPT

            agent = Agent(model=model, system_prompt=sys_prompt, tools=tools)
            res = agent(user_input)
            return res.message["content"][0]["text"]

    except Exception as e:
        logger.error("Unhandled error", exc_info=True)
        return f"[runtime error] {type(e).__name__}: {e}"

if __name__ == "__main__":
    app.run()

Overwriting runtime_app.py


In [3]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()


response = agentcore_runtime.configure(
    entrypoint="runtime_app.py",
    execution_role='arn:aws:iam::140083316867:role/oyaizu-bedrock-agentcore-runtime',
    auto_create_ecr=True,
    auto_create_s3=True,
    requirements_file=None,

    # ★ ランタイム → ハンドラーへヘッダを渡す Allowlist（これが超重要）
    request_header_configuration={
        "requestHeaderAllowlist": [
            "Authorization",  # OAuth 前方伝播に必須
            # 必要なら他のカスタムヘッダも追加
        ]
    },
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": ["30t82a8cdaq9j52pjf25gsmt44"],  # App Client ID
            "discoveryUrl": "https://cognito-idp.us-east-1.amazonaws.com/us-east-1_2Lu48C2MQ/.well-known/openid-configuration",
        }
    },
)

print("Configured:", response)

Entrypoint parsed: file=/home/sagemaker-user/runtime_app.py, bedrock_agentcore_name=runtime_app
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: runtime_app
Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: Dockerfile
Generated .dockerignore: /home/sagemaker-user/.dockerignore
Keeping 'runtime_app' as default agent
Bedrock AgentCore configured: /home/sagemaker-user/.bedrock_agentcore.yaml


Configured: config_path=PosixPath('/home/sagemaker-user/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/home/sagemaker-user/Dockerfile') dockerignore_path=PosixPath('/home/sagemaker-user/.dockerignore') runtime='Docker' runtime_type=None region='us-east-1' account_id='140083316867' execution_role='arn:aws:iam::140083316867:role/oyaizu-bedrock-agentcore-runtime' ecr_repository=None auto_create_ecr=True s3_path=None auto_create_s3=False memory_id=None network_mode='PUBLIC' network_subnets=None network_security_groups=None network_vpc_id=None


In [4]:
launch_result = agentcore_runtime.launch()  # CodeBuildでビルド＆デプロイ
print("Launched agent ARN:", launch_result.agent_arn)

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'runtime_app' to account 140083316867 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: runtime_app
ECR repository available: 140083316867.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-runtime_app
Using execution role from config: arn:aws:iam::140083316867:role/oyaizu-bedrock-agentcore-runtime
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 140083316867.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-runtime_app


Getting or creating CodeBuild execution role for agent: runtime_app
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-053e9d2563
Reusing existing CodeBuild execution role: arn:aws:iam::140083316867:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-053e9d2563
Using dockerignore.template with 46 patterns for zip filtering
Uploaded source to S3: runtime_app/source.zip
Updated CodeBuild project: bedrock-agentcore-runtime_app-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.0s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.3s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOURCE completed in 14.5s
🔄 BUILD started (total: 24s)
✅ BUILD completed in 45.5s
🔄 POST_BUILD started (total: 69s)
✅ POST_BUILD completed in 47.7s
🔄 COMPLETED started (total: 117s)
✅ COMPLETED completed in 1.0s
🎉 CodeBuild completed successfully in 1m 58s
CodeBuild completed successfully
CodeB

Launched agent ARN: arn:aws:bedrock-agentcore:us-east-1:140083316867:runtime/runtime_app-PE5KNe9iaL


In [5]:
st = agentcore_runtime.status()
print("Status:", st.endpoint.get("status"))

Retrieved Bedrock AgentCore status for: runtime_app


Status: READY


In [6]:
import requests
import json

CLIENT_ID = "30t82a8cdaq9j52pjf25gsmt44"
CLIENT_SECRET = "105fjfhd5n28qi5uk9r2kmo6vbibj6nfqrf7uju011srk66g3amd"
TOKEN_URL = "https://my-domain-c74k3de1.auth.us-east-1.amazoncognito.com/oauth2/token"

def fetch_access_token(client_id, client_secret, token_url):
  response = requests.post(
    token_url,
    data="grant_type=client_credentials&client_id={client_id}&client_secret={client_secret}".format(client_id=client_id, client_secret=client_secret),
    headers={'Content-Type': 'application/x-www-form-urlencoded'}
  )

  return response.json()['access_token']

def list_tools(gateway_url, access_token):
  headers = {
      "Content-Type": "application/json",
      "Authorization": f"Bearer {access_token}"
  }

  payload = {
      "jsonrpc": "2.0",
      "id": "list-tools-request",
      "method": "tools/list"
  }

  response = requests.post(gateway_url, headers=headers, json=payload)
  return response.json()

# Example usage
gateway_url = "https://oyaizu-gateway-dypv7pbiue.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
access_token = fetch_access_token(CLIENT_ID, CLIENT_SECRET, TOKEN_URL)
tools = list_tools(gateway_url, access_token)
print(json.dumps(tools, indent=2))

{
  "jsonrpc": "2.0",
  "id": "list-tools-request",
  "result": {
    "tools": [
      {
        "inputSchema": {
          "type": "object",
          "properties": {
            "id": {
              "description": "\u30e6\u30fc\u30b6\u30fcID\uff08\u4f8b: USER_001\uff09",
              "type": "string"
            },
            "username": {
              "description": "\u30e6\u30fc\u30b6\u30fc\u540d\uff08\u4f8b: taro\uff09",
              "type": "string"
            }
          },
          "required": [
            "id",
            "username"
          ]
        },
        "name": "target-quick-start-xv2t9k___DefaultTool",
        "description": "\u30c7\u30d5\u30a9\u30eb\u30c8\u30c4\u30fc\u30eb\u306e\u8aac\u660e"
      }
    ]
  }
}


In [7]:
access_token
print(type(access_token))

<class 'str'>


In [18]:
import uuid
session_id = str(uuid.uuid4())
resp = agentcore_runtime.invoke(
    {"prompt": "emailについて教えてください。id:USER_001, username:takumi"},
    bearer_token=access_token
)
print(resp['response'])

Using JWT authentication


"[runtime error] could not resolve MCP tool name."
